# MF-SSDM: CPU vs GPU spectral-radius benchmark on Colab

**Setup:** Runtime → Change runtime type → **Julia** + a GPU.

⚠️ **GPU choice matters — this is a double-precision (FP64) workload:**
| GPU | FP64 rate | verdict |
|---|---|---|
| T4 (free) | 1/32 (~0.25 TF) | no faster than a workstation card |
| L4 | 1/64 (~0.5 TF) | poor for FP64 |
| **A100** (Pro / compute units) | full rate (9.7 TF, 1.5 TB/s) | **the one to use** |

The kernel is grid-sync/bandwidth bound at d=2, so expect ~3–10× over a Quadro P4000 on A100 (more for larger-dimensional problems), not the raw 60× FLOP ratio.

In [ ]:
run(`nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv`)

In [ ]:
# ~5-10 min on first run (package resolution + CUDA artifact download)
import Pkg
Pkg.add(url="https://github.com/bachrathyd/StochasticSemiDiscretizationMethod.jl.git",
        rev="Collocation-based-RK_steps")
Pkg.add(["CUDA", "StaticArrays", "Plots"])
using StochasticSemiDiscretizationMethod, CUDA, StaticArrays, LinearAlgebra, Printf
CUDA.versioninfo()

In [ ]:
# fully periodic delay-stochastic Mathieu (A(t), B(t), α(t), β(t) all periodic)
const Am=3.0; const εm=2.0; const Bm=0.5; const ζm=0.1; const αm=0.1
const τm=2π;  const Pm=4π
function createProblem()
    AMxfun(t) = @SMatrix [0. 1.; -(Am + εm*cos(0.5t)) -2ζm]
    BMxfun(t) = @SMatrix [0. 0.; Bm*(1+0.4cos(0.5t)) 0.]
    αMxfun(t) = @SMatrix [0. 0.; -αm*(Am + εm*cos(0.5t)) -αm*2ζm]
    βMxfun(t) = @SMatrix [0. 0.; αm*Bm*(1+0.4cos(0.5t)) 0.]
    LDDEProblem(ProportionalMX(AMxfun), [DelayMX(τm, BMxfun)],
                [stCoeffMX(1, ProportionalMX(αMxfun))],
                [stCoeffMX(1, DelayMX(τm, βMxfun))],
                Additive(2), [stAdditive(1, Additive(@SVector [0., 0.]))])
end
mapping(p) = DiscreteMapping_M2_MF(
    StochasticSemiDiscretizationMethod.calculateResults(
        createProblem(), SemiDiscretization(2, Pm/p), τm, n_steps=p))

# correctness gate: CPU == GPU
dm = mapping(32)
ρc = spectralRadiusOfMapping_MF(dm); ρg = spectralRadiusOfMapping_GPU(dm)
@printf("CPU %.12f  GPU %.12f  rel %.1e\n", ρc, ρg, abs(ρc-ρg)/ρc)
@assert abs(ρc-ρg)/ρc < 1e-8

In [ ]:
using Plots
timeit(f) = (t0=time(); v=f(); (time()-t0, v))
function timeit3(f); t,v=timeit(f); for _ in 1:2; t2,_=timeit(f); t=min(t,t2); end; (t,v); end

ps = [64, 128, 256, 512, 1024, 2048, 4096, 6144, 8192]   # A100: all feasible
const CPU_CAP = 120.0     # drop the (weak) Colab vCPU beyond this per solve
rows = NamedTuple[]
cpu_stopped = false
for p in ps        # resolution-outer; plot updates every level
    dm = mapping(p)
    t_cpu = NaN; ρc = NaN
    if !cpu_stopped
        t_cpu, ρc = timeit3(() -> spectralRadiusOfMapping_MF(dm))
        t_cpu > CPU_CAP && (global cpu_stopped = true)
    end
    t_gpu, ρg = timeit3(() -> spectralRadiusOfMapping_GPU(dm))
    dis = isnan(ρc) ? NaN : abs(ρc-ρg)/ρc
    push!(rows, (p=p, t_cpu=t_cpu, t_gpu=t_gpu, ρ=isnan(ρc) ? ρg : ρc, dis=dis))
    @printf("p=%5d CPU %8.3fs GPU %8.3fs mismatch %.1e\n", p, t_cpu, t_gpu, dis)
    plt = plot([r.p for r in rows], [max(r.t_cpu,1e-4) for r in rows], marker=:circle,
               label="CPU (Colab vCPU)", xscale=:log10, yscale=:log10,
               xlabel="p", ylabel="wall time [s]", legend=:topleft)
    plot!(plt, [r.p for r in rows], [r.t_gpu for r in rows], marker=:star5,
          label="GPU (zero-sync)")
    display(plt)
end

In [ ]:
# save results; the CSV is ALSO printed so it can be copy-pasted back into the
# local repo as benchmark/colab_cpu_gpu.csv (then run benchmark/combined_wp.jl)
gpuname = strip(read(`nvidia-smi --query-gpu=name --format=csv,noheader`, String))
open("colab_cpu_gpu.csv","w") do io
    println(io,"# gpu: ", gpuname)
    println(io,"p,t_cpu,t_gpu,rho")
    for r in rows; @printf(io,"%d,%.4f,%.4f,%.12f\n",r.p,r.t_cpu,r.t_gpu,r.ρ); end
end
println("===== colab_cpu_gpu.csv =====")
println(read("colab_cpu_gpu.csv", String))
println("===== end =====")